# LTSA-60 versus legacy baseline: reproducible metric diagnostic

This notebook re-runs the read-only database evidence used in the portable report.

In [1]:
import pandas as pd
from sqlalchemy import text
from mstat_project.utils import get_db_engine

PARAMS = {'baseline_run_id': '075f26f2-e43e-452a-9854-08d75fccdce1', 'ltsa_run_id': '68f1b6de-f2b0-4faa-a674-e6e19ccc9735', 'ltsa_source_run_id': '07335dac-a57c-46f9-994f-4790af799221'}
engine = get_db_engine()
connection = engine.connect()
transaction = connection.begin()
connection.execute(text('SET TRANSACTION READ ONLY'))
print('Connected in a read-only transaction.')


Connected in a read-only transaction.


## Exact run and checkpoint match

In [2]:
RUN_SQL = "WITH baseline AS (\n    SELECT\n        'Legacy Cox baseline'::text AS model,\n        r.run_id::text AS run_id,\n        'All follow-up; baseline-to-outcome time'::text AS task,\n        'Event-averaged Cox partial NLL'::text AS loss_definition,\n        quoted.epoch AS quoted_epoch,\n        quoted.validation_loss::double precision AS quoted_validation_loss,\n        quoted.validation_c_index::double precision AS quoted_validation_c_index,\n        r.best_epoch AS selected_epoch,\n        selected.validation_loss::double precision AS selected_validation_loss,\n        selected.validation_c_index::double precision AS selected_validation_c_index,\n        r.test_c_index::double precision AS test_c_index,\n        r.train_patients,\n        r.validation_patients,\n        r.test_patients,\n        r.train_events,\n        r.validation_events,\n        r.test_events\n    FROM _ml.baseline_runs AS r\n    JOIN _ml.baseline_epoch_metrics AS quoted\n      ON quoted.run_id = r.run_id AND quoted.epoch = 15\n    JOIN _ml.baseline_epoch_metrics AS selected\n      ON selected.run_id = r.run_id AND selected.epoch = r.best_epoch\n    WHERE r.run_id = :baseline_run_id\n), ltsa AS (\n    SELECT\n        'LTSA at 60 months'::text AS model,\n        r.run_id::text AS run_id,\n        'At risk after month 60; residual time'::text AS task,\n        'Mean discrete-hazard NLL + next-visit MSE'::text AS loss_definition,\n        quoted.epoch AS quoted_epoch,\n        quoted.validation_total_loss::double precision AS quoted_validation_loss,\n        quoted.validation_c_index::double precision AS quoted_validation_c_index,\n        r.best_epoch AS selected_epoch,\n        selected.validation_total_loss::double precision AS selected_validation_loss,\n        selected.validation_c_index::double precision AS selected_validation_c_index,\n        r.test_c_index::double precision AS test_c_index,\n        r.train_patients,\n        r.validation_patients,\n        r.test_patients,\n        r.train_events,\n        r.validation_events,\n        r.test_events\n    FROM _ml.survival_runs AS r\n    JOIN _ml.survival_epoch_metrics AS quoted\n      ON quoted.run_id = r.run_id AND quoted.epoch = 45\n    JOIN _ml.survival_epoch_metrics AS selected\n      ON selected.run_id = r.run_id AND selected.epoch = r.best_epoch\n    WHERE r.run_id = :ltsa_run_id\n)\nSELECT * FROM baseline\nUNION ALL\nSELECT * FROM ltsa\nORDER BY quoted_epoch"
runs = pd.read_sql_query(text(RUN_SQL), connection, params=PARAMS)
print(runs.to_string(index=False))


              model                               run_id                                    task                           loss_definition  quoted_epoch  quoted_validation_loss  quoted_validation_c_index  selected_epoch  selected_validation_loss  selected_validation_c_index  test_c_index  train_patients  validation_patients  test_patients  train_events  validation_events  test_events
Legacy Cox baseline 075f26f2-e43e-452a-9854-08d75fccdce1 All follow-up; baseline-to-outcome time            Event-averaged Cox partial NLL            15                4.386823                   0.819633              12                   4.22424                     0.821138      0.816884            1843                  396            397           187                 41           41
  LTSA at 60 months 68f1b6de-f2b0-4faa-a674-e6e19ccc9735   At risk after month 60; residual time Mean discrete-hazard NLL + next-visit MSE            45                1.104910                   0.552356              35       

## Validation trajectories and metric decoupling

In [3]:
EPOCH_SQL = "SELECT\n    'Legacy Cox baseline'::text AS model,\n    epoch,\n    validation_loss::double precision AS validation_loss,\n    validation_c_index::double precision AS validation_c_index\nFROM _ml.baseline_epoch_metrics\nWHERE run_id = :baseline_run_id\nUNION ALL\nSELECT\n    'LTSA at 60 months'::text AS model,\n    epoch,\n    validation_total_loss::double precision AS validation_loss,\n    validation_c_index::double precision AS validation_c_index\nFROM _ml.survival_epoch_metrics\nWHERE run_id IN (:ltsa_source_run_id, :ltsa_run_id)\nORDER BY model, epoch"
epochs = pd.read_sql_query(text(EPOCH_SQL), connection, params=PARAMS)
milestones = []
for model, group in epochs.groupby('model'):
    min_loss = group.loc[group['validation_loss'].idxmin()]
    max_c = group.loc[group['validation_c_index'].idxmax()]
    milestones.append({'model': model, 'minimum-loss epoch': int(min_loss['epoch']), 'minimum loss': min_loss['validation_loss'], 'C at minimum loss': min_loss['validation_c_index'], 'maximum-C epoch': int(max_c['epoch']), 'maximum C': max_c['validation_c_index']})
print(pd.DataFrame(milestones).to_string(index=False))


              model  minimum-loss epoch  minimum loss  C at minimum loss  maximum-C epoch  maximum C
  LTSA at 60 months                   3      0.824026           0.615183               35   0.688482
Legacy Cox baseline                  12      4.224240           0.821138               12   0.821138


## Cohort support and comparable pairs

In [4]:
COHORT_SQL = "WITH cohorts AS (\n    SELECT\n        'Legacy Cox baseline'::text AS model,\n        ptid,\n        split,\n        observed_time_months,\n        event_observed\n    FROM _ml.baseline_run_patients\n    WHERE run_id = :baseline_run_id\n    UNION ALL\n    SELECT\n        'LTSA at 60 months'::text AS model,\n        ptid,\n        split,\n        observed_time_months,\n        event_observed\n    FROM _ml.survival_run_patients\n    WHERE run_id = :ltsa_run_id\n), counts AS (\n    SELECT\n        model,\n        split,\n        count(*) AS patients,\n        count(*) FILTER (WHERE event_observed) AS events\n    FROM cohorts\n    WHERE split IN ('validation', 'test')\n    GROUP BY model, split\n), pairs AS (\n    SELECT\n        earlier.model,\n        earlier.split,\n        count(*) AS comparable_pairs\n    FROM cohorts AS earlier\n    JOIN cohorts AS later\n      ON later.model = earlier.model\n     AND later.split = earlier.split\n     AND earlier.event_observed\n     AND earlier.observed_time_months < later.observed_time_months\n    WHERE earlier.split IN ('validation', 'test')\n    GROUP BY earlier.model, earlier.split\n)\nSELECT counts.model, counts.split, counts.patients, counts.events,\n       pairs.comparable_pairs\nFROM counts\nJOIN pairs USING (model, split)\nORDER BY counts.split DESC, counts.model"
cohorts = pd.read_sql_query(text(COHORT_SQL), connection, params=PARAMS)
print(cohorts.to_string(index=False))


              model      split  patients  events  comparable_pairs
Legacy Cox baseline validation       396      41              6642
  LTSA at 60 months validation        74       9               382
Legacy Cox baseline       test       397      41              6444
  LTSA at 60 months       test        76       8               363


## Scan timing and split-overlap audit

In [5]:
AUDIT_SQL = "WITH shared AS (\n    SELECT\n        legacy.ptid,\n        legacy.split AS baseline_split,\n        ltsa.split AS ltsa_split,\n        legacy.image_id AS baseline_image_id,\n        ltsa.selected_image_id AS ltsa_image_id,\n        legacy.observed_time_months AS baseline_observed_time,\n        legacy.event_observed AS baseline_event,\n        images.months_since_baseline_image::double precision AS baseline_scan_month\n    FROM _ml.survival_run_patients AS ltsa\n    JOIN _ml.baseline_run_patients AS legacy\n      ON legacy.run_id = :baseline_run_id\n     AND legacy.ptid = ltsa.ptid\n    JOIN _core.core_image_set AS images\n      ON images.ptid = legacy.ptid\n     AND images.image_id::text = legacy.image_id\n    WHERE ltsa.run_id = :ltsa_run_id\n), all_baseline_scans AS (\n    SELECT\n        patients.observed_time_months,\n        images.months_since_baseline_image::double precision AS scan_month\n    FROM _ml.baseline_run_patients AS patients\n    JOIN _core.core_image_set AS images\n      ON images.ptid = patients.ptid\n     AND images.image_id::text = patients.image_id\n    WHERE patients.run_id = :baseline_run_id\n), baseline_scan_correlation AS (\n    SELECT corr(scan_month, observed_time_months) AS value\n    FROM all_baseline_scans\n), baseline_early_events AS (\n    SELECT count(*) AS value\n    FROM _ml.baseline_run_patients\n    WHERE run_id = :baseline_run_id\n      AND split = 'validation'\n      AND event_observed\n      AND observed_time_months <= 60\n)\nSELECT\n    count(*) AS shared_patients,\n    count(*) FILTER (WHERE baseline_image_id = ltsa_image_id) AS same_selected_image,\n    count(*) FILTER (WHERE baseline_scan_month > 60) AS baseline_scans_after_month_60,\n    percentile_cont(0.5) WITHIN GROUP (ORDER BY baseline_scan_month)\n        AS median_baseline_scan_month,\n    count(*) FILTER (WHERE baseline_split <> ltsa_split) AS changed_split,\n    count(*) FILTER (\n        WHERE ltsa_split = 'validation' AND baseline_split = 'validation'\n    ) AS ltsa_validation_also_baseline_validation,\n    count(*) FILTER (WHERE ltsa_split = 'validation') AS ltsa_validation_patients,\n    max(baseline_early_events.value) AS baseline_validation_events_by_month_60,\n    max(baseline_scan_correlation.value) AS full_baseline_scan_outcome_correlation\nFROM shared\nCROSS JOIN baseline_scan_correlation\nCROSS JOIN baseline_early_events\nGROUP BY ()"
audit = pd.read_sql_query(text(AUDIT_SQL), connection, params=PARAMS)
print(audit.to_string(index=False))


 shared_patients  same_selected_image  baseline_scans_after_month_60  median_baseline_scan_month  changed_split  ltsa_validation_also_baseline_validation  ltsa_validation_patients  baseline_validation_events_by_month_60  full_baseline_scan_outcome_correlation
             529                   34                            495                       89.69            216                                        18                        74                                      34                                0.985525


## Interpretation

The raw losses are incommensurate, the C-indices are supported by very different numbers of events/pairs, and the cohorts differ in target clock, MRI information boundary, and validation membership. The correct next experiment is the paired fixed-landmark workflow.

In [6]:
transaction.rollback()
connection.close()
engine.dispose()
print('Closed the read-only transaction.')


Closed the read-only transaction.
